# **Step1_AI면접관 Agent v1.0**

## **1. 환경준비**

### (1) 구글 드라이브

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project_genai)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### (2) 라이브러리

In [6]:
!pip install -r /content/drive/MyDrive/genai/requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.7/437.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

### (3) OpenAI API Key 확인
* api_key.txt 파일에 다음의 키를 등록하세요.
    * OPENAI_API_KEY
    * NGROK_AUTHTOKEN

In [7]:
import pandas as pd
import numpy as np
import os
import ast
import fitz  # PyMuPDF
from docx import Document
import random
import openai
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from typing import Annotated, Literal, Sequence, TypedDict

from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, START, END

In [10]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/genai/'
# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

In [11]:
print(os.environ['OPENAI_API_KEY'][:30])

sk-proj-ZI6eWCjE449Sj-44sV-CjG


## **2. App.py**

* 아래 코드에, Step1 혹은 고도화 된 Step2 파일 코드를 붙인다.
    * 라이브러리
    * 함수들과 그래프
* Gradio 코드는 그대로 사용하거나 일부 수정 가능

In [12]:
%%writefile app.py

####### 여러분의 함수와 클래스를 모드 여기에 붙여 넣읍시다. #######
## 1. 라이브러리 로딩 ---------------------------------------------
import pandas as pd
import numpy as np
import os
import openai
import random
import ast
import fitz
from docx import Document

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from typing import Annotated, Literal, Sequence, TypedDict, List, Dict
from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.output_parsers import CommaSeparatedListOutputParser
from langgraph.graph import StateGraph, START, END
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

## ---------------- 1단계 : 사전준비 ----------------------

# 1) 파일 입력 --------------------
def extract_text_from_file(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        doc = fitz.open(file_path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        return text
    elif ext == ".docx":
        doc = Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        raise ValueError("지원하지 않는 파일 형식입니다. PDF 또는 DOCX만 허용됩니다.")

# 2) State 선언 --------------------
class InterviewState(TypedDict):
    # 고정 정보
    resume_text: str
    resume_summary: str
    resume_keywords: List[str]
    question_strategy: Dict[str, Dict]

    # 인터뷰 로그
    current_question: str
    current_answer: str
    current_strategy: str
    complete_strategy: Dict[str, bool]
    conversation: List[Dict[str, str]]
    evaluation : List[Dict[str, str]]
    next_step : str

    # InterviewState에 다음 항목 추가
    # 피드백 평가 상태
    reflection_status: str  # "정상" 또는 "재평가 필요"
    reflection_retry_count: int  # 재평가 시도 횟수
    reflection_retry_counts: List[int]  # 각 질문별 재평가 횟수 리스트

# 3) resume 분석 --------------------
def analyze_resume(state: InterviewState) -> InterviewState:
    # 여기에 코드를 완성합니다.
    resume_text = state["resume_text"]
    llm = ChatOpenAI(model_name = 'gpt-4o-mini')

    prompt1 = ChatPromptTemplate.from_messages([
        ("system", "너는 대기업 HR 부서에서 수많은 이력서를 검토해온 면접 전문가야."),
        ("human", """다음 이력서 및 자기소개서 내용에서 질문을 뽑기 위한 중요한 내용을
                      10문장 정도로 요약을 해줘(요약시 ** 기호는 사용하지 말것):\n\n{resume_text}""")
    ])

    formatted_messages1 = prompt1.format_messages(
        resume_text=resume_text
    )
    response1 = llm.invoke(formatted_messages1)
    resume_summary = response1.content

    prompt2 = ChatPromptTemplate.from_messages([
        ("system", "너는 대기업 HR 부서에서 수많은 이력서를 검토해온 면접 전문가야."),
        ("human", """다음 이력서 및 자기소개서내용에서 질문을 뽑기 위한 중요한 핵심 키워드를 5~10개 추출해줘.
                      도출한 핵심 키워드만 쉼표로 구분해줘:\n\n{resume_text}"""),
        ("system", "{format_instructions}")
    ])

    parser = CommaSeparatedListOutputParser()

    formatted_messages2 = prompt2.format_messages(
        resume_text=resume_text,
        format_instructions=parser.get_format_instructions()
    )

    response2 = llm.invoke(formatted_messages2)
    resume_keywords = parser.parse(response2.content)

    # return 코드는 제공합니다.
    return {
        **state,
        "resume_summary": resume_summary,
        "resume_keywords": resume_keywords,
    }

# 4) 질문 전략 수립 --------------------

def generate_question_strategy(state: InterviewState) -> InterviewState:
    # 여기에 코드를 완성합니다.
    resume_summary = state["resume_summary"]
    resume_keywords = state["resume_keywords"]
    llm = ChatOpenAI(model_name = 'gpt-4o-mini')

    schemas = [
        ResponseSchema(name="경력 및 경험", description="경력 및 경험 기반 질문 전략. 방향성과 예시 질문 리스트 포함"),
        ResponseSchema(name="동기 및 커뮤니케이션", description="동기 및 커뮤니케이션 기반 질문 전략. 방향성과 예시 질문 리스트 포함"),
        ResponseSchema(name="논리적 사고", description="논리적 사고 기반 질문 전략. 방향성과 예시 질문 리스트 포함"),
    ]

    parser = StructuredOutputParser.from_response_schemas(schemas)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "너는 대기업의 면접관이야. 이력서를 바탕으로 맞춤형 질문 전략을 수립하는 전문가야."),
        ("human", """다음 이력서 요약과 키워드를 바탕으로 다음 3가지 질문 전략을 만들어줘.

                      각 전략은 다음과 같은 key를 사용해:
                      - "경력 및 경험": 경력 및 경험 기반 질문
                      - "동기 및 커뮤니케이션": 동기 및 커뮤니케이션 기반 질문
                      - "논리적 사고": 논리적 사고 기반 질문

                      각 전략은 아래와 같이 구성돼야 해:
                      - 질문 방향: 지원자의 어느 경험을 바탕으로 무엇을 파악하기 위해 질문을 정할지에 대한 방향성
                      - 예시 질문: 방향성과 관련한 예시 질문 2개 (문장 리스트)

                      분야:
                      1. 경력 및 경험 질문 (경력 및 경험)
                      2. 동기 및 커뮤니케이션 질문 (동기 및 커뮤니케이션)
                      3. 논리적 사고 질문 (논리적 사고)

                      [이력서 요약]
                      {resume_summary}

                      [키워드]
                      {resume_keywords}

                      응답은 반드시 JSON 형식으로 해줘. """),
        ("system", "{format_instructions}")
    ])

    formatted_messages = prompt.format_messages(
        format_instructions=parser.get_format_instructions(),
        resume_summary=resume_summary,
        resume_keywords=", ".join(resume_keywords)
    )

    response = llm.invoke(formatted_messages)
    strategy_dict = parser.parse(response.content)

   # return 코드는 제공합니다.
    return {
        **state,
        "question_strategy": strategy_dict
    }

# 5) 1단계 하나로 묶기 --------------------

def preProcessing_Interview(file_path: str) -> InterviewState:
    # 여기에 코드를 완성합니다.
    resume_text = extract_text_from_file(file_path)

    state: InterviewState = {
        "resume_text": resume_text,
        "resume_summary": '',
        "resume_keywords": [],
        "question_strategy": {},

        "current_question": '',
        "current_answer": '',
        "current_strategy": '',
        "complete_strategy": {"경력 및 경험":False, "동기 및 커뮤니케이션":False, "논리적 사고":False},
        "conversation": [],
        "evaluation": [],
        "next_step" : '',

        "reflection_status": "정상",  # "정상" 또는 "재평가 필요"
        "reflection_retry_count": 0,  # 재평가 시도 횟수 (초기값 0)
        "reflection_retry_counts": [] # # 각 질문별 재평가 횟수 리스트
    }

    state  = analyze_resume(state)

    state = generate_question_strategy(state)

    selected_question = state["question_strategy"]["경력 및 경험"]["예시 질문"][0]

    # return 코드는 제공합니다.
    return {
        **state,
        "current_question": selected_question,
        "current_strategy": "경력 및 경험"
    }


## ---------------- 2단계 : 면접 Agent ----------------------

# 1) 답변 입력 --------------------
def update_current_answer(state: InterviewState, user_answer: str) -> InterviewState:
    return {
        **state,
        "current_answer": user_answer.strip()
    }

# 2) 답변 평가 --------------------
def evaluate_answer(state: InterviewState) -> InterviewState:
    from langchain.chat_models import ChatOpenAI
    from langchain.output_parsers import ResponseSchema, StructuredOutputParser
    from langchain.prompts import ChatPromptTemplate

    current_question = state.get("current_question", "")
    current_answer = state.get("current_answer", "")
    conversation = state.get("conversation", [])
    llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

    schemas = [
        ResponseSchema(name="relevance", description="질문과의 관련성 평가 (상/중/하 중 하나)"),
        ResponseSchema(name="specificity", description="답변의 구체성 평가 (상/중/하 중 하나)")
    ]

    parser = StructuredOutputParser.from_response_schemas(schemas)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """너는 대기업 면접관이야. 면접자의 답변을 아래 기준에 따라 평가해줘.

                      1. 질문과의 연관성 (상/중/하)
                          - 상: 질문의 핵심 의도에 정확히 부합하며, 전반적인 내용을 명확히 다룸
                          - 중: 질문과 관련은 있지만 핵심 포인트가 부분적으로 누락됨
                          - 하: 질문과 관련이 약하거나 엉뚱한 내용 중심

                      2. 답변의 구체성 (상/중/하)
                          - 상: 명확한 근거, 수치, 경험, 사례 등을 포함해 구체적으로 설명함
                          - 중: 전체적으로 추상적이지만, 일부 구체적인 사례나 표현이 있음
                          - 하: 모호하고 일반적인 표현 위주이며, 구체적인 정보가 거의 없음

                      ✅ [출력 형식 예시]
                      질문과의 연관성: 상
                      답변의 구체성: 중
                      평가 코멘트: (간결하게 요약된 평가, 50자 내외)"""),
        ("human", """다음은 면접관의 질문과 그에 대한 지원자의 답변이야.

                      질문: {current_question}
                      답변: {current_answer}

                      위 질문에 대한 답변이 적절한지 다음의 평가 항목에서 상/중/하로 평가해줘.

                      1. 질문과의 관련성
                      2. 답변의 구체성

                      응답은 반드시 JSON 형식으로 해줘. """),
        ("system", "{format_instructions}")
    ])

    formatted_messages = prompt.format_messages(
        format_instructions=parser.get_format_instructions(),
        current_question=current_question,
        current_answer=current_answer
    )

    try:
        response = llm.invoke(formatted_messages)
        parsed = parser.parse(response.content)
    except Exception as e:
        print("평가 오류:", e)
        parsed = {"relevance": "하", "specificity": "하"}

    evaluation = [
        {"항목": "관련성", "등급": parsed.get("relevance", "하")},
        {"항목": "구체성", "등급": parsed.get("specificity", "하")}
    ]

    state["evaluation"] = evaluation

    if state.get("reflection_status") == "정상":
        state["conversation"].append({
            "질문": current_question,
            "답변": current_answer,
            "평가": evaluation
        })
        state["reflection_status"] = ""

    return state

def feedback_evaluate(state: InterviewState) -> InterviewState:
    from langchain.chat_models import ChatOpenAI
    from langchain.prompts import ChatPromptTemplate

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    current_question = state.get("current_question", "")
    current_answer = state.get("current_answer", "")
    evaluation = state.get("evaluation", [])

    # 등급 추출
    relevance = next((e["등급"] for e in evaluation if e["항목"] == "관련성"), "없음")
    specificity = next((e["등급"] for e in evaluation if e["항목"] == "구체성"), "없음")

    retry = state.get("reflection_retry_count", 0)
    if retry >= 2:
        state["reflection_status"] = "정상"
        return state

    prompt = ChatPromptTemplate.from_messages([
        ("system", """너는 면접 평가 보조 AI야. 다음 질문과 답변, 그리고 그에 대한 평가는 사람이 한 평가 결과야.

이 사람이 내린 평가가 적절한지를 아래 기준에 따라 판단해줘:

- 답변 내용과 평가 등급이 어울리는지 확인
- 등급이 너무 후하거나 박하지 않은지 판단
- 평가가 타당하면 "정상", 이상하거나 재검토가 필요하면 "재평가 필요"라고만 대답해줘

<입력 정보>
질문: {question}
답변: {answer}
평가:
- 질문과의 연관성: {relevance}
- 답변의 구체성: {specificity}

("system",
아래 정보가 타당한지 판단해줘. 결과는 반드시 '정상' 또는 '재평가 필요' 둘 중 하나만 딱 한 줄로 출력해.
다른 말 하지 마. 예: 정상)
""")
    ])

    formatted = prompt.format_messages(
        question=current_question,
        answer=current_answer,
        relevance=relevance,
        specificity=specificity
    )

    try:
        response = llm.invoke(formatted)
        feedback = response.content.strip().replace(".", "")
    except Exception as e:
        print("피드백 평가 오류:", e)
        feedback = "재평가 필요"

    state["reflection_status"] = "정상" if "정상" in feedback else "재평가 필요"
    state["reflection_retry_count"] = retry + 1
    state.setdefault("reflection_retry_counts", []).append(retry)

    return state

# 3) 인터뷰 진행 검토 --------------------
def decide_next_step(state: InterviewState) -> InterviewState:
    # 여기에 코드를 완성합니다.
    question_strategy = state.get("question_strategy", {})
    evaluation = state.get("evaluation", {})
    current_strategy = state.get("current_strategy", "")
    complete_strategy = state.get("complete_strategy", {})

    # 모든 전략 영역 커버
    flag = True
    next_strategy = ""
    for strategy in question_strategy:
        if complete_strategy[strategy] == False:
            flag = False
            next_strategy=strategy

    if flag == True:
      return {
          **state,
          "current_strategy": current_strategy,
          "next_step": "end"
      }

    # 대화 횟수
    num_interactions = len(state["conversation"])
    # 전체 질문&답변 5회
    if num_interactions > 5:
        return {
            **state,
            "next_step": "end"
        }

    complete_strategy[current_strategy] = True

    # 최근 평가에 따라
    flag = True
    for e in (evaluation):
        if e["등급"] == '하':
          flag = False

    if flag == False:
      return {
          **state,
          "next_step": "additional_question"
      }
    else:
      return {
          **state,
          "current_strategy": next_strategy,
          "next_step": "next_strategy"
      }

# 4) 질문 생성 --------------------
# import 추가
from langchain.document_loaders import CSVLoader

# 벡터DB 및 리트리버 생성함수
def generate_Retriever_from_VectorDB(path: str,file_name: str):
  # CSV 파일 로드
  csv_path = path + file_name
  csv_loader = CSVLoader(file_path=csv_path)
  documents_csv = csv_loader.load()

  # 벡터 DB 정의 및 생성
  embedding = OpenAIEmbeddings(model="text-embedding-3-small")
  vectorstore = Chroma.from_documents(documents_csv, embedding,
                                      persist_directory=path+"/db")
  # 리트리버 생성
  return vectorstore.as_retriever(search_kwargs={"k": 3})

path = '/content/drive/MyDrive/genai/'
file_name = "similar_questions 1.csv"
retriever = generate_Retriever_from_VectorDB(path,file_name)

def generate_question(state: InterviewState) -> InterviewState:
    # 여기에 코드를 완성합니다.
    llm = ChatOpenAI(model="gpt-4o-mini")

    ### 고도화 부분
    resume_keywords = ", ".join(state.get("resume_keywords", []))

    question_strategy = state.get("question_strategy", {})
    current_strategy = state.get("current_strategy", "")
    strategy = question_strategy[current_strategy]['질문 방향']

    resume_summary = state.get("resume_summary", "")
    current_question = state.get("current_question", "")
    current_answer = state.get("current_answer", "")
    evaluation = state.get("evaluation", [])

    query = f"{resume_keywords}, {strategy}"
    docs = retriever.invoke(query)
    similar_questions = [doc.page_content for doc in docs]

    # 유사 질문 제대로 받아왔는지 검증
    # for question in similar_questions:
    #     print("질문")
    #     print(question)

    ###

    # 프롬프트 수정
    prompt = ChatPromptTemplate.from_messages([
        ("system", """ 당신은 전문 면접관이며 인터뷰 질문을 설계하는 AI입니다.
다음은 추가 질문을 생성하기 위해 참조할 중요한 정보입니다.
- 이력서 요약: {resume_summary}
- 이력서 키워드: {resume_keywords}
- 질문 전략({current_strategy}): {strategy}
- 이전 질문: {current_question}
- 답변: {current_answer}
- 평가: {evaluation}
- 유사 질문 예시: {similar_questions}

위 정보를 기반으로 지원자의 사고력, 문제 해결 방식, 혹은 기술적 깊이를 더 확인할 수 있는 심화 인터뷰 질문을
유사 질문 예시를 참고하여 기반으로 새로운 질문을 한 가지 생성해주세요.
구체적이고, 지원자의 대답을 확장시킬 수 있는 질문이어야 합니다. 또한 날카로운 질문이어야 합니다. 질문은 한 문장으로 생성합니다.
'왜', '어떻게', '무엇을 기준으로' 형태로 시작할 수 있다면 그 방식으로 시작하세요
""")
        ])

    messages = prompt.format_messages(
        resume_summary=resume_summary,
        resume_keywords=", ".join(resume_keywords),
        current_strategy=current_strategy,
        current_question= current_question,
        current_answer=current_answer,
        question_strategy=question_strategy,
        strategy=strategy,
        evaluation=evaluation,
        similar_questions = similar_questions
    )

    response = llm.invoke(messages)

    # return 코드는 제공합니다.
    return {
        **state,
        "current_question": response.content.strip(),
        "current_answer": ""
    }

# 5) 인터뷰 피드백 보고서 --------------------
def summarize_interview(state: InterviewState) -> InterviewState:

    conversation = state["conversation"]
    reflection_retry_counts = state.get("reflection_retry_counts", [])
    llm = ChatOpenAI(model_name="gpt-4o")

    # 대화와 평가 데이터 변환 함수 정의
    def format_conversation(entries):
        text = ""
        for idx, entry in enumerate(entries, 1):
            text += f"질문{idx}: {entry['질문']}\n"
            text += f"답변{idx}: {entry['답변']}\n"
            evaluation_str = ", ".join([f"{eval_item['항목']}: {eval_item['등급']}" for eval_item in entry["평가"]])
            text += f"평가: {evaluation_str}\n\n"
        return text

    # 1. 전략별로 대화 그룹화
    strategy_groups = {}
    for entry in conversation:
        strategy = entry.get("question_strategy", "기타")
        if strategy not in strategy_groups:
            strategy_groups[strategy] = []
        strategy_groups[strategy].append(entry)

    # 2. 전략별 분석 수행
    strategy_analyses = {}
    strategy_prompt = ChatPromptTemplate.from_messages([
        ("system", """인터뷰 피드백 전문가로서, 주어진 전략 영역에 대한 면접자의 답변들을 분석해주세요.
                    다음 형식으로 구조화된 분석을 제공해주세요:

                    - 답변 스타일: 면접자의 답변 방식, 표현 특징 등을 요약
                    - 강점: 이 전략 영역에서 드러난 주요 강점 (2-3개)
                    - 약점: 이 전략 영역에서 보완이 필요한 부분 (1-2개)

                    분석은 객관적이고 건설적이며 구체적인 피드백을 담아야 합니다."""),
        ("human", """
                [전략 영역]
                {strategy}

                [관련 질문-답변]
                {conversations}
                """)
    ])

    for strategy, entries in strategy_groups.items():
        conversation_text = format_conversation(entries)
        messages = strategy_prompt.format_messages(
            strategy=strategy,
            conversations=conversation_text
        )

        response = llm.invoke(messages)
        strategy_analyses[strategy] = response.content

    # 3. 평가 점수 통계 분석
    score_mapping = {"상": 3, "중": 2, "하": 1}
    all_evaluations = [eval_item for entry in conversation for eval_item in entry["평가"]]

    # 평가 항목별 점수 추출 함수
    def get_avg_score(eval_type):
        scores = [score_mapping.get(item["등급"], 0) for item in all_evaluations if item["항목"] == eval_type]
        return sum(scores) / len(scores) if scores else 0

    avg_relevance = get_avg_score("관련성")
    avg_specificity = get_avg_score("구체성")

    # 재평가 통계 분석
    avg_retry = sum(reflection_retry_counts) / len(reflection_retry_counts) if reflection_retry_counts else 0
    max_retry = max(reflection_retry_counts) if reflection_retry_counts else 0
    evaluation_reliability = "높음" if avg_retry < 0.5 else ("중간" if avg_retry < 1.5 else "낮음")

    # 4. 종합 피드백 생성
    # 전략별 분석을 문자열로 변환
    strategy_text = "\n\n".join([f"[{strategy}]\n{analysis}" for strategy, analysis in strategy_analyses.items()])

    overall_prompt = ChatPromptTemplate.from_messages([
        ("system", """인터뷰 평가 전문가로서, 전체 인터뷰 데이터를 분석하여 종합적인 피드백 보고서를 작성해주세요.
                    다음 섹션을 포함해야 합니다:

                    - 전체 인상: 인터뷰 전반에 대한 인상적인 요약 (2-3문장)
                    - 주요 강점: 가장 두드러진 3가지 강점
                    - 개선 필요사항: 가장 중요한 2-3가지 개선점
                    - 종합 평가: 인터뷰 전체에 대한 간결한 평가와 조언 (3-4문장)

                    피드백은 구체적이고 건설적이며 실행 가능한 조언을 포함해야 합니다.

                    평가 신뢰도가 '낮음'으로 표시된 경우, 이 점을 고려하여 더 신중한 피드백을 제공하세요."""),
        ("human", """
                [인터뷰 데이터]
                {conversation_data}

                [평가 통계]
                관련성 평균 점수: {avg_relevance}/3
                구체성 평균 점수: {avg_specificity}/3
                평가 신뢰도: {reliability} (재평가 평균 시도: {avg_retry}, 최대 시도: {max_retry})

                [전략별 분석]
                {strategy_analyses}
                """)
    ])

    messages = overall_prompt.format_messages(
        conversation_data=format_conversation(conversation),
        avg_relevance=round(avg_relevance, 1),
        avg_specificity=round(avg_specificity, 1),
        reliability=evaluation_reliability,
        avg_retry=round(avg_retry, 1),
        max_retry=max_retry,
        strategy_analyses=strategy_text
    )

    overall_feedback = llm.invoke(messages).content

    # 5. 최종 보고서 생성
    report = f"""
    # 인터뷰 피드백 보고서

    ## 평가 신뢰도 정보
    - 평가 신뢰도: **{evaluation_reliability}**
    - 재평가 평균 시도: {round(avg_retry, 1)}회
    - 최대 재평가 시도: {max_retry}회

    ## 종합 피드백
    {overall_feedback}

    ## 전략별 상세 분석
    """

    for strategy, analysis in strategy_analyses.items():
        report += f"""
    ### {strategy} 전략
    {analysis}
    """

    report += """
    ## 인터뷰 기록 요약
    """

    for idx, entry in enumerate(conversation, 1):
        retry_info = ""
        if idx - 1 < len(reflection_retry_counts) and reflection_retry_counts[idx - 1] > 0:
            retry_info = f" (재평가 시도: {reflection_retry_counts[idx - 1]}회)"

        report += f"""
    ### 질문-답변 #{idx}{retry_info}
    **질문**: {entry['질문']}

    **답변**: {entry['답변']}

    **평가**: {', '.join([f"{eval_item['항목']}: {eval_item['등급']}" for eval_item in entry["평가"]])}
    """

    # 콘솔에 출력
    print("\n" + "=" * 80)
    print("인터뷰 피드백 보고서")
    print("=" * 80)
    print(report)
    print("=" * 80)

    return {
        **state,
        "interview_report": report
    }

# 6) Agent --------------------
# 분기 판단 함수 (기존)
def route_next(state: InterviewState) -> Literal["generate", "summarize"]:
    state = decide_next_step(state)
    return "summarize" if state["next_step"] == "end" else "generate"

# 추가: feedback_evaluate 결과에 따른 분기
def route_feedback(state: InterviewState) -> Literal["evaluate", "decide"]:
    return "evaluate" if state.get("reflection_status") != "정상" else "decide"

# 그래프 정의 시작
builder = StateGraph(InterviewState)

# 노드 추가
builder.add_node("evaluate", evaluate_answer)
builder.add_node("feedback", feedback_evaluate)  # ← 추가됨
builder.add_node("decide", decide_next_step)
builder.add_node("generate", generate_question)
builder.add_node("summarize", summarize_interview)

# 노드 연결
builder.add_edge(START, "evaluate")
builder.add_edge("evaluate", "feedback")  # ← feedback 노드로 연결
builder.add_conditional_edges("feedback", route_feedback, {
    "decide": "decide",
    "evaluate": "evaluate"
})
builder.add_conditional_edges("decide", route_next, {
    "generate": "generate",
    "summarize": "summarize"
})
builder.add_edge("generate", END)
builder.add_edge("summarize", END)

# 컴파일
graph = builder.compile()
#-------------------------------------------------------------------


########### 다음 코드는 제공되는 gradio 코드 입니다.################

import gradio as gr
import tempfile

# 세션 상태 초기화 함수
def initialize_state():
    return {
        "state": None,
        "interview_started": False,
        "interview_ended": False,
        "chat_history": []
    }

# 파일 업로드 후 인터뷰 초기화
def upload_and_initialize(file_obj, session_state):
    if file_obj is None:
        return session_state, "파일을 업로드해주세요."

    # Gradio는 file_obj.name 이 파일 경로야
    file_path = file_obj.name

    # 인터뷰 사전 처리
    state = preProcessing_Interview(file_path)
    session_state["state"] = state
    session_state["interview_started"] = True

    # 첫 질문 저장
    first_question = state["current_question"]
    session_state["chat_history"].append(["🤖 AI 면접관", first_question])

    return session_state, session_state["chat_history"]

# 답변 처리 및 다음 질문 생성
def chat_interview(user_input, session_state):
    if not session_state["interview_started"]:
        return session_state, "먼저 이력서를 업로드하고 인터뷰를 시작하세요."

    # (1) 사용자 답변 저장
    session_state["chat_history"].append(["🙋‍♂️ 지원자", user_input])
    session_state["state"] = update_current_answer(session_state["state"], user_input)

    # (2) Agent 실행 (평가 및 다음 질문 or 종료)
    next_step = session_state["state"]["next_step"] if "next_step" in session_state["state"] else ""

    # Agent 실행
    result = graph.invoke(session_state["state"])
    session_state["state"] = result

    # (3) 종료 여부 판단
    if session_state["state"]["next_step"] == "end":
        session_state["interview_ended"] = True

        # 먼저 summarize_interview 함수 실행하여 보고서 생성
        if "interview_report" not in session_state["state"]:
            print("인터뷰 보고서 생성 중...")
            session_state["state"] = summarize_interview(session_state["state"])

        # 인터뷰 보고서 있으면 표시
        if "interview_report" in session_state["state"]:
            print("인터뷰 보고서 출력:", session_state["state"]["interview_report"][:100] + "...")
            final_report = session_state["state"]["interview_report"]
            session_state["chat_history"].append(["🤖 AI 면접관", final_report])
        else:
            # 보고서 없을 경우 대화 요약만
            conversation_summary = "✅ 인터뷰가 종료되었습니다!\n\n"
            conversation_summary += "## 인터뷰 대화 요약\n"
            for i, turn in enumerate(session_state["state"]["conversation"]):
                conversation_summary += f"\n**[질문 {i+1}]** {turn['질문']}\n**[답변 {i+1}]** {turn['답변']}\n"
                if "평가" in turn:
                    eval_result = turn["평가"]
                    eval_text = ", ".join([f"{item['항목']}: {item['등급']}" for item in eval_result])
                    conversation_summary += f"_평가 - {eval_text}_\n"

            session_state["chat_history"].append(["🤖 AI 면접관", conversation_summary])

        return session_state, session_state["chat_history"], gr.update(value="")

    else:
        next_question = session_state["state"]["current_question"]
        session_state["chat_history"].append(["🤖 AI 면접관", next_question])
        return session_state, session_state["chat_history"], gr.update(value="")

# Gradio 인터페이스 구성
with gr.Blocks() as demo:
    session_state = gr.State(initialize_state())

    gr.Markdown("# 🤖 AI 면접관 \n이력서를 업로드하고 인터뷰를 시작하세요!")

    with gr.Row():
        file_input = gr.File(label="이력서 업로드 (PDF 또는 DOCX)")
        upload_btn = gr.Button("인터뷰 시작")

    chatbot = gr.Chatbot()
    user_input = gr.Textbox(show_label=False, placeholder="답변을 입력하고 Enter를 누르세요.")

    upload_btn.click(upload_and_initialize, inputs=[file_input, session_state], outputs=[session_state, chatbot])
    user_input.submit(chat_interview, inputs=[user_input, session_state], outputs=[session_state, chatbot, user_input])

# 실행
demo.launch(share=True)

Writing app.py


## **3. 실행**

In [13]:
!python app.py

/content/app.py:410: LangChainDeprecationWarning: Importing CSVLoader from langchain.document_loaders is deprecated. Please replace deprecated imports:

>> from langchain.document_loaders import CSVLoader

with new imports of:

>> from langchain_community.document_loaders import CSVLoader
You can use the langchain cli to **automatically** upgrade many imports. Please see documentation here <https://python.langchain.com/docs/versions/v0_2/>
  from langchain.document_loaders import CSVLoader
/content/app.py:420: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings(model="text-embedding-3-small")
/content/app.py:794: UserWarning: You have not specified a valu